# Train SALI-PyCCE Prototype in Google Colab

This notebook trains the compact SALI-style signal-to-image model on synthetic CPMG data.

Recommended Colab setup: **Runtime → Change runtime type → GPU**.

If the repo is private, use a GitHub personal access token or upload the ZIP manually.

In [ ]:
# Check GPU
!nvidia-smi || true

import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## Clone the repository

For a public repo, use the simple clone command. For a private repo, create a fine-grained GitHub token with read access to this repo, then paste it when prompted.

In [ ]:
# Option A: public repo
# !git clone https://github.com/userwhe/sali-pycce-prototype.git

# Option B: private repo using token
from getpass import getpass
import os, subprocess, textwrap

if not os.path.exists('/content/sali-pycce-prototype'):
    token = getpass('GitHub token: ')
    url = f'https://{token}@github.com/userwhe/sali-pycce-prototype.git'
    subprocess.run(['git', 'clone', url], check=True)
    del token

%cd /content/sali-pycce-prototype

In [ ]:
# Install the project. Colab already usually has torch installed.
!pip install -q -e .[dev]

# Optional: try PyCCE install. The current repo uses analytic backend by default.
# !pip install -q pycce

In [ ]:
# Run tests
!pytest -q

## Tiny smoke-test training

This should finish quickly and verifies the full training loop.

In [ ]:
!python -m sali_pycce.train \
  --train-samples 512 \
  --val-samples 128 \
  --epochs 3 \
  --batch-size 32 \
  --signal-points 256 \
  --max-spins 5 \
  --device cuda \
  --out checkpoints/colab_smoke.pt

## Larger training run

Use this after the smoke test works. This is still smaller than the paper-scale dataset, but more meaningful than the toy run.

In [ ]:
!python -m sali_pycce.train \
  --train-samples 20000 \
  --val-samples 2000 \
  --epochs 20 \
  --batch-size 128 \
  --signal-points 512 \
  --max-spins 10 \
  --device cuda \
  --out checkpoints/colab_medium.pt

In [ ]:
# Evaluate
!python -m sali_pycce.evaluate \
  --checkpoint checkpoints/colab_medium.pt \
  --samples 500 \
  --signal-points 512 \
  --max-spins 10 \
  --threshold 0.25 \
  --device cuda

In [ ]:
# Visualize one prediction
!python examples/predict_one.py \
  --checkpoint checkpoints/colab_medium.pt \
  --signal-points 512 \
  --max-spins 10 \
  --out runs/prediction_demo.png

from IPython.display import Image, display
display(Image('runs/prediction_demo.png'))

## Save checkpoint to Google Drive

Colab files disappear after the runtime ends, so save trained weights to Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/sali_checkpoints
!cp checkpoints/*.pt /content/drive/MyDrive/sali_checkpoints/
!ls -lh /content/drive/MyDrive/sali_checkpoints/